In [0]:
display(dbutils.fs.ls("/mnt/raw"))

In [0]:
#### setting widgets here to test - usually passed from ADF parmaeters 

source_folder = dbutils.widgets.text("source_folder", "")
file_format = dbutils.widgets.text("file_format", "")
bronze_table_name = dbutils.widgets.text("bronze_table_name", "")

uc_catalog = dbutils.widgets.text("uc_catalog", "")
uc_schema = dbutils.widgets.text("uc_schema", "")

In [0]:
source_folder = dbutils.widgets.get("source_folder")
file_format = dbutils.widgets.get("file_format")
bronze_table_name = dbutils.widgets.get("bronze_table_name")

In [0]:
base_raw_path = "/mnt/raw/"
base_bronze_path = "/mnt/bronze/"
base_checkpoint_path = "/mnt/bronze/checkpoints/"
base_schema_path = "/mnt/bronze/schemas/" 

In [0]:
current_source_path = f"{base_raw_path}{source_folder}/"
current_bronze_table_path = f"{base_bronze_path}{bronze_table_name}/"
current_checkpoint_path = f"{base_checkpoint_path}{bronze_table_name}/"
current_schema_path = f"{base_schema_path}{bronze_table_name}/"

In [0]:
# intializing Unity Catalog settings 

uc_catalog = dbutils.widgets.get("uc_catalog")
uc_schema = dbutils.widgets.get("uc_schema")

uc_catalog_table = f"{uc_catalog}.{uc_schema}.{bronze_table_name}"

In [0]:
print(f"Processing source folder: {source_folder}")
print(f"Ingesting from: {current_source_path}")
print(f"File format: {file_format}")
print(f"Target Bronze table name: {bronze_table_name}")
print(f"Target Bronze table path: {current_bronze_table_path}")
print(f"Autoloader Checkpoint: {current_checkpoint_path}")
print(f"Autoloader Schema: {current_schema_path}")
print(f"Current Unity catalog table used for the run: {uc_catalog_table}")

In [0]:
from pyspark.sql.types import *

autoloader_options = {}

if file_format == "json":
    customer_schema = StructType([
        StructField("customer_id", StringType(), True),
        StructField("first_name", StringType(), True),
        StructField("last_name", StringType(), True),
        StructField("date_of_birth", StringType(), True), # Can cast to DateType later
        StructField("gender", StringType(), True),
        StructField("address", StringType(), True),
        StructField("phone_number", StringType(), True),
        StructField("email", StringType(), True),
        StructField("registration_date", StringType(), True), # Can cast to DateType later
        StructField("preferred_contact_method", StringType(), True)
    ])

    autoloader_options = {
        'cloudFiles.format': 'json',
        'cloudFiles.schemaLocation': current_schema_path,
        'cloudFiles.schemaEvolutionMode': 'addNewColumns', # https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema
        'multiline' : 'true',
        'schema': customer_schema,
        'cloudFiles.inferSchema': 'true'
    }

elif file_format == "csv":
    autoloader_options = {
        'cloudFiles.format': 'csv',
        'cloudFiles.schemaLocation': current_schema_path,
        'cloudFiles.schemaEvolutionMode': 'addNewColumns', # https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema
        'delimiter' : ',',
        'header': 'true',
        'inferSchema': 'true'
    }

elif file_format == "parquet":
    autoloader_options = {
        'cloudFiles.format': 'parquet',
        'cloudFiles.schemaLocation': current_schema_path,
        'cloudFiles.schemaEvolutionMode': 'addNewColumns', # https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema
    }

else:
    print("Invalid File Format, please pass JSON, CSV or Parquet")

    


In [0]:
print(autoloader_options)

In [0]:
df_raw = (
    spark
    .readStream
    .format("cloudFiles")
    .options(**autoloader_options)
    .load(current_source_path)
)

In [0]:
(
df_raw
.writeStream
.outputMode("append")
.option("checkpointLocation", f"{current_checkpoint_path}_write/")
.trigger(availableNow=True)
.toTable(uc_catalog_table)
)

In [0]:
print(f"Successfully ingested data from '{current_source_path}' to Unity Catalog Bronze table: '{uc_catalog_table}' at physical path: '{current_bronze_table_path}'")


In [0]:
display(spark.sql(f"select * from {uc_catalog_table}"))